# Sparkle diagnostic — where does the speckle enter the pipeline?

The grad(x)² and kinematic fields show pixel-level 'sparkle'
(isolated extreme pixels).  The store-vs-live consistency checks in
the field-validation notebooks already rule out the
float32→64→32 round-trip (residuals ~1e-7 relative, pure rounding),
so the sparkle must be *created* during the calculation.  This
notebook shows every stage of the dependency chain side by side to
identify the first stage where it appears:

- **frontal_structure** chains exercise
  `native_gradient.calculate_native_gradient_tracer` +
  `native_gradient.grad_squared`;
- **kinematic** chains exercise
  `native_gradient.calculate_jacobian`.

Figure layout — one figure per final field:

| | col 1 (raw) | ... | col n (final) |
|---|---|---|---|
| row 1 | store, full region | | |
| row 2 | store, 200×200 km zoom | | |
| row 3 | live recompute, full region | | |
| row 4 | live recompute, zoom | | |

Sequential colormaps only (diverging maps can hide speckle in their
white midpoint); one shared norm per column so store and live are
directly comparable.  Stages that are never saved (gradient /
Jacobian components) show a placeholder in the store rows.

## Section 1 — RUN the SURF pipeline (both subsets)

Existing stores for this date are skipped unless `--clobber` is
added, so re-running is a cheap no-op.

In [ ]:
# Section 1: run the SURF pipeline for both subsets + timestep.
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset frontal_structure \
    --run_id $RUN_ID

!generate-global \
    --config ../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset kinematic \
    --run_id $RUN_ID

## Section 2 — LOAD both stores + the shared grid

In [ ]:
# Section 2: store readers (frontal_structure + kinematic) + grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
readers = {}
for _subset in ("frontal_structure", "kinematic"):
    _defn = get_subset_definition(PIPELINE, _subset)
    readers[_subset] = zarr_dataset.GlobalZarrDatasetReader(
        bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
        dataset_name=_defn["dataset_name"],
        date_prefix=DATE_PREFIX, fs=fs,
    )
    print(f"{_subset}: {readers[_subset].channel_names}")

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat
print(f"grid: XC {XC.shape}")

## Section 3 — Region selection + chains

`REGION` is selectable; any region with a `zoom` anchor works
(`gulf_stream`, `kuroshio`, `so_atlantic`, `eq_pacific`,
`kerguelen`).  Re-run from here after changing it.

In [ ]:
# Section 3: pick the region; define the dependency chains.
from dbof.plotting import regions

REGION       = "gulf_stream"   # <-- change me, re-run from here
ZOOM_HALF_KM = 100.0           # 200x200 km zoom box

_zoomable = [n for n, r in regions.REGIONS.items() if "zoom" in r]
assert REGION in _zoomable, f"pick one of {_zoomable}"
print(f"region: {REGION}  (options: {_zoomable})")

# Dependency chains, raw -> intermediates -> final.  Frontal chains
# exercise calculate_native_gradient_tracer + grad_squared; the
# kinematic chains exercise calculate_jacobian.
CHAINS = {
    # frontal_structure
    "gradtheta2": ["Theta", "dTheta_dx", "dTheta_dy", "gradtheta2"],
    "gradsalt2":  ["Salt", "dSalt_dx", "dSalt_dy", "gradsalt2"],
    "gradeta2":   ["Eta", "dEta_dx", "dEta_dy", "gradeta2"],
    "gradrho2":   ["rho_theta", "drho_dx", "drho_dy", "gradrho2"],
    "gradb2":     ["buoyancy", "db_dx", "db_dy", "gradb2"],
    # kinematic
    "relative_vorticity": ["U", "V", "dv_dx", "du_dy",
                           "relative_vorticity"],
    "divergence":         ["U", "V", "du_dx", "dv_dy",
                           "divergence"],
    "strain_n":           ["U", "V", "du_dx", "dv_dy", "strain_n"],
    "strain_s":           ["U", "V", "dv_dx", "du_dy", "strain_s"],
    "strain_mag":         ["strain_n", "strain_s", "strain_mag"],
    "okubo_weiss":        ["strain_mag", "relative_vorticity",
                           "okubo_weiss"],
}
STAGES = sorted({s for c in CHAINS.values() for s in c})
print(f"{len(CHAINS)} chains, {len(STAGES)} distinct stages")

## Section 4 — LIVE recomputation (raw → intermediates → finals)

Same loaders and same code as `generate-global`: one OSN snapshot,
every chain stage recomputed lazily in float64, batch-stitched, and
sliced to `REGION` immediately (bounded memory).  The finals here
are the *live* versions of the store channels — per the consistency
checks they agree with the store to ~1e-7, so any sparkle visible
in the store rows must also appear here.

In [ ]:
# Section 4a: snapshot + lazy live fields for every chain stage.
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt", "Eta", "U", "V"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}")
for _v in ("Theta", "Salt", "Eta", "U", "V"):
    print(f"  raw {_v}: {ds_merge[_v].dtype}")

# frontal chain: tracer gradients (calculate_native_gradient_tracer)
# and their grad_squared finals.
rho = calculate_fields.potential_density(ds_merge)
b = calculate_fields.buoyancy_of_field(ds_merge)
gTh = ng.calculate_native_gradient_tracer(
    ds_merge.Theta, ds_merge, grid=xgrid)
gS = ng.calculate_native_gradient_tracer(
    ds_merge.Salt, ds_merge, grid=xgrid)
gE = ng.calculate_native_gradient_tracer(
    ds_merge.Eta, ds_merge, grid=xgrid)
gR = ng.calculate_native_gradient_tracer(rho, ds_merge, grid=xgrid)
gB = ng.calculate_native_gradient_tracer(b, ds_merge, grid=xgrid)

# kinematic chain: rotated velocities + Jacobian (calculate_jacobian)
# and the finals computed from the SAME shared Jacobian.
u_east, v_north = calculate_fields.geographic_velocity(
    ds_merge, xgrid)
J = calculate_fields.compute_velocity_jacobian(ds_merge, xgrid)
s_mag, s_n, s_s = calculate_fields.strain(
    ds_merge, xgrid, jacobian=J)

live_map = {
    # frontal raw + intermediates
    "Theta": ds_merge["Theta"], "Salt": ds_merge["Salt"],
    "Eta": ds_merge["Eta"], "rho_theta": rho, "buoyancy": b,
    "dTheta_dx": gTh[0], "dTheta_dy": gTh[1],
    "dSalt_dx": gS[0], "dSalt_dy": gS[1],
    "dEta_dx": gE[0], "dEta_dy": gE[1],
    "drho_dx": gR[0], "drho_dy": gR[1],
    "db_dx": gB[0], "db_dy": gB[1],
    # frontal finals (live)
    "gradtheta2": calculate_fields.grad_theta2(ds_merge, xgrid),
    "gradsalt2": calculate_fields.grad_salt2(ds_merge, xgrid),
    "gradeta2": calculate_fields.grad_eta2(ds_merge, xgrid),
    "gradrho2": calculate_fields.grad_rho2(ds_merge, xgrid),
    "gradb2": calculate_fields.grad_b2(ds_merge, xgrid),
    # kinematic raw + Jacobian components
    "U": u_east, "V": v_north,
    "du_dx": J.du_dx, "du_dy": J.du_dy,
    "dv_dx": J.dv_dx, "dv_dy": J.dv_dy,
    # kinematic finals (live, shared Jacobian)
    "relative_vorticity": calculate_fields.relative_vorticity(
        ds_merge, xgrid, jacobian=J),
    "divergence": calculate_fields.divergence(
        ds_merge, xgrid, jacobian=J),
    "strain_n": s_n, "strain_s": s_s, "strain_mag": s_mag,
    "okubo_weiss": calculate_fields.okubo_weiss_parameter(
        ds_merge, xgrid, jacobian=J),
}
missing = [s for s in STAGES if s not in live_map]
assert not missing, f"live_map is missing stages: {missing}"
print(f"{len(live_map)} lazy live fields ready")

In [ ]:
# Section 4b: batch-stitch the live fields, slice to REGION only.
BATCH = 4
mask = {"_land_mask": (ds_merge.hFacC == 0)}
names = [s for s in live_map if s in STAGES]
live_region = {}
for i0 in range(0, len(names), BATCH):
    grp = names[i0:i0 + BATCH]
    ds_conv = ds_raw.assign({n: live_map[n] for n in grp})[grp]
    chw = stitch_and_mask(ds_conv, grp, mask)
    for k, n in enumerate(grp):
        live_region[n] = regions.select_region(chw[k], XC, YC,
                                               REGION)
    del chw
    print(f"stitched + sliced: {grp}")
print(f"live fields ready: {sorted(live_region)}")

## Section 5 — STORE slices

Store channels sliced to `REGION`.  Stages that are not store
channels (gradient / Jacobian components, `rho_theta`) stay
live-only and show a placeholder in the store rows.

In [ ]:
# Section 5: slice the store channels used by any chain to REGION.
store_region = {}
for _subset, _rd in readers.items():
    for ch in _rd.channel_names:
        if ch in STAGES and ch not in store_region:
            arr = _rd.get_channel_snapshot(ch)
            store_region[ch] = regions.select_region(arr, XC, YC,
                                                     REGION)
            del arr
print(f"stored stages   : {sorted(store_region)}")
print(f"live-only stages: {sorted(set(STAGES) - set(store_region))}")

## Section 6 — Stage-by-stage sparkle figures

How to read each figure: scan the columns left to right and find
the **first** column whose zoom rows show isolated extreme pixels.

- Sparkle already in column 1 (raw) → it's in the model data.
- First appears in the gradient / Jacobian columns → created by the
  native-grid differencing.
- Only in the final column → created by the squaring / combination
  step (which amplifies: a 10x outlier becomes 100x).

Store and live rows share the column norm — if they sparkle
identically, precision handling is ruled out (consistent with the
~1e-7 consistency-check residuals).

In [ ]:
# Section 6 helper: one call per final field.
from dbof.plotting.sparkle_grids import sparkle_stage_grid


def sparkle_figure(field, cmap="viridis"):
    """Render the 4-row stage grid for one final field.

    Inputs: field (str) — key into CHAINS; cmap (str) — sequential
    colormap name.  Outputs: displays the figure.
    Generated by LH and Claude
    """
    chain = CHAINS[field]
    store_xyz = {s: store_region.get(s) for s in chain}
    live_xyz = {s: live_region[s] for s in chain}
    sparkle_stage_grid(
        chain, store_xyz, live_xyz, REGION,
        half_km=ZOOM_HALF_KM, cmap=cmap,
        suptitle=(f"{field} \u2014 sparkle trace "
                  f"({REGION}, store vs live)"),
    )
    plt.show()

### gradtheta2

Sparkle candidate #1 (see the validation Figure 1).
Columns: raw Theta → dTheta_dx → dTheta_dy → |∇Θ|².

In [ ]:
sparkle_figure("gradtheta2")

### gradsalt2

In [ ]:
sparkle_figure("gradsalt2")

### gradeta2

In [ ]:
sparkle_figure("gradeta2")

### gradrho2

rho_theta is live-only (the store carries `density` =
rho_theta − 1000).

In [ ]:
sparkle_figure("gradrho2")

### gradb2

In [ ]:
sparkle_figure("gradb2")

### relative_vorticity

Kinematic chain: rotated U, V → Jacobian components → ζ.
Sequential colormap on the signed field — speckle shows as
isolated bright/dark dots.

In [ ]:
sparkle_figure("relative_vorticity")

### divergence

In [ ]:
sparkle_figure("divergence")

### strain_n

In [ ]:
sparkle_figure("strain_n")

### strain_s

In [ ]:
sparkle_figure("strain_s")

### strain_mag

In [ ]:
sparkle_figure("strain_mag")

### okubo_weiss

In [ ]:
sparkle_figure("okubo_weiss")

## Findings

*(fill in after running)*

- First stage showing sparkle, per chain:
- Store vs live identical? (expected: yes)
- Interpretation: